# The Price Is Right - Week 7 - Day 3 (RL Training)

## Training with Reinforcement Learning (PPO)

In this notebook, we will train the model using Proximal Policy Optimization (PPO) to directly minimize the pricing error.

Our reward function will be: `Reward = -|Predicted Price - True Price|`

If you are using LITE_MODE=True, then please run this on a free T4 box.

If you are using LITE_MODE=False, then please use a paid A100 or L4 with high memory.

## 1. Install Libraries

We install the necessary libraries for Reinforcement Learning on LLMs:
*   `trl`: Transformer Reinforcement Learning library from Hugging Face, containing the `PPOTrainer`.
*   `peft`: For Parameter-Efficient Fine-Tuning (LoRA).
*   `bitsandbytes`: For 4-bit quantization to fit the model in GPU memory.
*   `accelerate` & `transformers`: Core Hugging Face libraries.

**Note**: We adhere to `numpy<2.0` and `trl==0.9.6` to ensure compatibility. We perform a clean install to avoid binary incompatibility issues (ValueError: numpy.dtype size changed).

In [ ]:
# Aggressively uninstall potentially conflicting libraries first
!pip uninstall -y numpy torch torchvision torchaudio trl transformers peft accelerate bitsandbytes
# Install numpy < 2.0 explicitly
!pip install "numpy<2.0"
# Install compatible versions of torch ecosystem and other libs
!pip install torch torchvision torchaudio transformers datasets peft "trl==0.9.6" bitsandbytes accelerate wandb

## 2. Imports and Configuration

Here we setup our environment and define the hyperparameters.

**Key Configurations:**
*   **LoRA**: We use Rank 32 (or 256 for full mode) adapters to efficiently train the model.
*   **PPO parameters**: We set a small learning rate (`1.41e-5`) and a batch size optimized for the GPU (128 for L4/T4 in Lite mode).
*   **L4 Optimization**: We automatically detect if `bfloat16` is supported (which it is on L4) to improve training stability and performance.

In [1]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import wandb
from peft import LoraConfig
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price-rl"
HF_USER = "Rodan009" # your HF name here!

LITE_MODE = True

DATA_USER = "Rodan009"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 80 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Hyper-parameters - PPO Training

LEARNING_RATE = 1.41e-5
# L4 has 24GB VRAM, but PPO overhead is high. Reducing batch sizes to prevent OOM.
BATCH_SIZE = 32 if LITE_MODE else 64
MINI_BATCH_SIZE = 4 if LITE_MODE else 8
GRADIENT_ACCUMULATION_STEPS = 1
MAX_NEW_TOKENS = 12
PPO_EPOCHS = 1

capability = torch.cuda.get_device_capability()
# L4 (and A100) supports bfloat16 (capability >= 8)
use_bf16 = capability[0] >= 8

# Tracking

LOG_TO_WANDB = True

## 3. Logins
We log in to Hugging Face to download the model and dataset, and to Weights & Biases for experiment tracking.

In [3]:
# Log in to HuggingFace
from getpass import getpass
if 'HF_TOKEN' in os.environ:
    hf_token = os.environ['HF_TOKEN']
else:
    #hf_token = userdata.get('HF_TOKEN')
    hf_token = getpass("Hugging Face Token: ")
login(hf_token, add_to_git_credential=True)

In [8]:
wandb_api_key = getpass("WandB API Key: ")
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rodan009 (rodan009-ai) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 4. Load Model and Tokenizer

We load the `Llama-3.2-3B` model.

**Important for PPO:**
We wrap the model with `AutoModelForCausalLMWithValueHead`. This adds an extra linear layer (the "Value Head") to the model that outputs a scalar value for each token. This value represents the **expected future reward**, which is crucial for the PPO algorithm to calculate advantages.

We also apply **LoRA** (Low-Rank Adaptation) configuration here to ensure we are fine-tuning efficiently.

In [9]:
if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

# Load Model with Value Head for PPO
# This wraps the base model to allow for value estimation

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    peft_config=LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=TARGET_MODULES,
    ),
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
# PPO usually works better with left padding for generation
tokenizer.padding_side = "left"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

## 5. Data Processing

We load the dataset which consists of prompts and completions.

**Crucial Step:**
We must extract the **True Price** from the `completion` field. The completion might look like `"Price is $12.34"`. We use regex to extract `12.34` so we can use it mathematically in our reward function.

In [10]:
def extract_price(text):
    # completion format is usually "123.00" or similar
    # We extract the first float number found
    match = re.search(r"[-+]?\d*\.\d+|\d+", text)
    return float(match.group()) if match else 0.0

def build_dataset(dataset_name, tokenizer):
    ds = load_dataset(dataset_name, split="train")
    
    # Filter out very long prompts to fit context
    original_columns = ds.column_names
    
    def preprocess_function(examples):
        new_examples = {
            "query": [],
            "input_ids": [],
            "true_price": []
        }
        for prompt, completion in zip(examples["prompt"], examples["completion"]):
             new_examples["query"].append(prompt)
             new_examples["input_ids"].append(tokenizer.encode(prompt, add_special_tokens=False))
             new_examples["true_price"].append(extract_price(completion))

        return new_examples

    ds = ds.map(
        preprocess_function,
        batched=True,
        remove_columns=original_columns,
    )
    
    ds = ds.filter(lambda x: len(x["input_ids"]) < 512)
    ds.set_format(type="torch")
    return ds

dataset = build_dataset(DATASET_NAME, tokenizer)
print(f"Loaded {len(dataset)} examples")
dataset[0]

Loaded 20000 examples


{'query': 'What does this cost to the nearest dollar?\n\nTitle: Schlage Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half)\nCategory: Door Hardware\nBrand: Schlage\nDescription: Interior half of a two-piece Andover knob with deadbolt in oil rubbed bronze.\nDetails: Includes knob and deadbolt; non-handed knob style; requires F58 to complete handle set; 4" minimum center-to-center door prep; Lifetime Mechanical and Finish Warranty.\n\nPrice is $',
 'input_ids': tensor([ 3923,  1587,   420,  2853,   311,   279, 24379, 18160,  1980,  3936,
            25, 50379,   425,  1628,  2017, 29958, 13934,   677,   449, 15371,
         53533,    11, 15895, 13134,  2788, 45967,   320, 86225, 26924,   340,
          6888,    25, 25166, 37865,   198, 28268,    25, 50379,   425,   198,
          5116,    25, 29958,  4376,   315,   264,  1403, 56964,  1628,  2017,
         59672,   449,  5710, 53533,   304,  5707, 67854, 40907,   627,  7955,
            25, 27044, 59672,   323,  5710,

## 6. Initialize PPO Trainer

We set up the `PPOTrainer`.

Note the `ref_model=None`. In PPO, we usually need a **Relationship to the Reference Model** (KL Divergence) to prevent the model from drifting too far from the original language distribution (which ensures it still speaks English).

By setting `ref_model=None`, `trl` automatically creates a reference model that shares layers with the active model but keeps the adapters frozen. This saves huge amounts of memory!

In [11]:
# Custom Collator to ensure we keep 'true_price' AND pad 'input_ids'
def collator(data):
    # data is a list of dicts
    # Separate 'true_price' (scalar) to handle it manually
    true_prices = [d["true_price"] for d in data]
    
    # Filter data for DataCollatorWithPadding: preserve ONLY model inputs
    # We must exclude 'query' (strings) because DataCollatorWithPadding crashes on strings
    # We only keep 'input_ids' and 'attention_mask' (if present)
    model_inputs = [{k: v for k, v in d.items() if k in ["input_ids", "attention_mask"]} for d in data]
    
    # Use DataCollatorWithPadding to automatically pad input_ids and attention_mask
    padding_collator = transformers.DataCollatorWithPadding(tokenizer)
    batch = padding_collator(model_inputs)
    
    # Add 'true_price' back to the batch as a tensor
    batch["true_price"] = torch.stack(true_prices)
    
    return batch

config = PPOConfig(
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    mini_batch_size=MINI_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    remove_unused_columns=False,
    gradient_checkpointing=True
)

ppo_trainer = PPOTrainer(
    config,
    model,
    ref_model=None, # None means using the initial model as reference (shared weights with adapter frozen)
    tokenizer=tokenizer,
    dataset=dataset,
    data_collator=collator
)

## 7. The PPO Training Loop

This is the core of Reinforcement Learning. Unlike SFT where we just feed data, here we iterate through these steps:

1.  **Rollout (Generation)**: We give the model a query (product description) and ask it to generate a completion (the price).
2.  **Reward Computation**: We parse the generated text to find the predicted price. We compare it to the `true_price`. The Reward is the negative distance: `Reward = -abs(Predicted - True)`.
3.  **PPO Step**: We call `ppo_trainer.step()`. This calculates the advantages and updates the model weights to maximize the expected reward, while keeping the KL divergence low (staying close to the reference model).
4.  **Logging**: We log the rewards to see if the model is actually learning to be more accurate.

In [12]:
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.eos_token_id,
    "max_new_tokens": MAX_NEW_TOKENS,
}

if LOG_TO_WANDB:
    wandb.init(project=PROJECT_NAME, name=RUN_NAME)

save_steps = 50
step_count = 0

for epoch in range(PPO_EPOCHS):
    print(f"Epoch {epoch+1}/{PPO_EPOCHS}")
    for batch in tqdm(ppo_trainer.dataloader):
        # PPOTrainer.generate expects a list of 1D tensors, not a 2D batch tensor
        query_tensors = [t for t in batch["input_ids"]]
        
        # 1. Generate responses
        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,
            **generation_kwargs
        )
        
        batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]
        
        # 2. Compute Rewards
        rewards = []
        predicted_prices = []
        errors = []
        
        for response, true_price in zip(batch["response"], batch["true_price"]):
            predicted_price = extract_price(response)
            predicted_prices.append(predicted_price)
            
            # Reward = Negative Absolute Error
            error = abs(predicted_price - true_price)
            reward = -error
            
            # Optional: Clip reward to avoid extreme values destabilizing training
            # reward = max(reward, -100.0) 
            
            rewards.append(torch.tensor(reward))
            errors.append(error)
            
        # 3. PPO Step
        # PPOTrainer.step also expects list of tensors
        stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
        
        # 4. Logging
        batch_reward = torch.stack(rewards).mean().item()
        batch_error = sum(errors) / len(errors)
        
        if LOG_TO_WANDB:
            wandb.log({
                "reward": batch_reward,
                "error": batch_error,
                "mean_true_price": sum(batch["true_price"]) / len(batch["true_price"]),
                "mean_predicted_price": sum(predicted_prices) / len(predicted_prices),
                **stats
            })
            
        step_count += 1
        if step_count % save_steps == 0:
            ppo_trainer.save_pretrained(PROJECT_RUN_NAME)
            # Optional: push to hub
            try:
                ppo_trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
            except Exception as e:
                print(f"Failed to push to hub: {e}")

# Save final model
ppo_trainer.save_pretrained(PROJECT_RUN_NAME)
ppo_trainer.model.push_to_hub(HUB_MODEL_NAME, private=True)
print(f"Saved to the hub: {HUB_MODEL_NAME}")
if LOG_TO_WANDB:
    wandb.finish()

Epoch 1/1


  0%|          | 0/625 [00:00<?, ?it/s]/tmp/ipython-input-211658058.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  rewards.append(torch.tensor(reward))
  0%|          | 2/625 [00:54<4:33:25, 26.33s/it]/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:1304: UserWarning: KL divergence is starting to become negative: -1.05 - this might be a precursor for failed training. sometimes this happens because the generation kwargs are not correctly set. Please make sure that the generation kwargs are set correctly, or review your training hyperparameters.
  warnings.warn(
  0%|          | 3/625 [01:23<4:49:52, 27.96s/it]/usr/local/lib/python3.12/dist-packages/trl/trainer/ppo_trainer.py:1304: UserWarning: KL divergence is starting to become negative: -1.89 - this might be a precursor for failed training. sometimes this happen

KeyboardInterrupt: 